# Phase 2: Fine-Tuning Open-Source Transformer Models
### DistilBERT & RoBERTa for Sequence Classification
**National Institute of Technology Sikkim** — Dept. of Computer Science & Engineering  
*Authors:* Pravesh Gedon Sharma (B240019) & Ankit Rawat (B320032)

---
### 🎯 Objectives of this Notebook:
1. **Implement Phase 2 Transformer Pipeline** (Section 5):
   - Model selection: **DistilBERT** (lightweight student baseline) & **RoBERTa** (bidirectional contextual encoder).
2. **Tokenization and Input Formatting** (Section 5.2):
   - Subword tokenization, dynamic batch padding, and truncation.
   - Handling long documents with chunking / sliding window.
3. **Fine-Tuning Loop & Optimization** (Section 5.4):
   - Cross-entropy loss minimization with AdamW optimizer.
   - Learning rate scheduling with linear warm-up (	imes 10^{-5}$) and early stopping on validation F1.
4. **Evaluation and Model Checkpointing**.

## 1. Setup & Device Configuration

In [ ]:
import sys
import os
sys.path.append("..")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Hardware device detection (Apple Silicon MPS, CUDA, or CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using compute device: {device}")

## 2. Dataset Preparation & PyTorch Dataset Class

In [ ]:
TRAIN_PATH = "../train.csv" if os.path.exists("../train.csv") else "train.csv"
df_raw = pd.read_csv(TRAIN_PATH)

# Sample subset for fine-tuning
SUBSET_SIZE = 2500
df_subset = df_raw.groupby("generated", group_keys=False).apply(
    lambda x: x.sample(SUBSET_SIZE, random_state=42)
).reset_index(drop=True)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_subset["text"].values,
    df_subset["generated"].values,
    test_size=0.20,
    random_state=42,
    stratify=df_subset["generated"].values
)

class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }

print(f"Prepared {len(train_texts)} train & {len(val_texts)} validation samples.")

## 3. Tokenizer and Transformer Model Loading
We use Hugging Face  to load DistilBERT ().

In [ ]:
MODEL_NAME = "distilbert-base-uncased"

try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    model.to(device)
    print(f"Loaded {MODEL_NAME} successfully.")
except Exception as e:
    print(f"HuggingFace transformers not initialized locally: {e}")
    print("To run this notebook with network access: pip install transformers accelerate")

## 4. Fine-Tuning Execution Loop (Section 5.4)
Using the AdamW optimizer with learning rate 	imes 10^{-5}$ and evaluation on validation F1.

In [ ]:
from torch.optim import AdamW
from tqdm.auto import tqdm
from src.evaluate import compute_classification_metrics

if "model" in locals():
    train_dataset = TextClassificationDataset(train_texts, train_labels, tokenizer)
    val_dataset = TextClassificationDataset(val_texts, val_labels, tokenizer)
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    epochs = 2

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        print(f"Epoch {epoch+1} Average Training Loss: {total_loss / len(train_loader):.4f}")
else:
    print("Skipping training loop (Transformers package not present in current environment).")

## 5. Sliding Window Chunking for Long Documents (Section 5.2)
Because transformers have token length limits (typically 512), long documents are split into overlapping chunks, and individual chunk probabilities are averaged.

In [ ]:
def classify_long_document(text, tokenizer, model, chunk_size=256, overlap=64):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    if len(tokens) <= chunk_size:
        chunks = [tokens]
    else:
        step = chunk_size - overlap
        chunks = [tokens[i:i + chunk_size] for i in range(0, len(tokens), step)]
    
    probs = []
    for chunk in chunks:
        inputs = tokenizer.prepare_for_model(chunk, max_length=chunk_size, padding="max_length", return_tensors="pt")
        with torch.no_grad():
            logits = model(inputs["input_ids"].to(device)).logits
            prob = torch.softmax(logits, dim=-1)[0, 1].item()
            probs.append(prob)
    return float(np.mean(probs))

print("Long document chunking logic defined.")

## 6. Key Takeaways & Summary
1. **Contextual Representations**: Unlike static TF-IDF, Transformer self-attention attends to long-range syntactic nuances and coherence.
2. **Practical Progression**: Follows the recommended trajectory from Section 5.4: TF-IDF + Logistic Regression -> Linear SVM -> DistilBERT fine-tuning -> RoBERTa fine-tuning.
3. **Final Comparison**: In Notebook , we evaluate all models on the held-out  dataset.